This notebook will conduct the preprocessing of the text data collected for the Kitwe project. The motivation is to uset this text data to classify the document as fake and genuine.  Before going into the modeling, we will do load the text data and do some basic text preprocessing needed before feeding the data into transformers. 

For using standard ML and RNN models, we already conducted text preprocessing like lowercasing, removing stop words, lemmatization, tokenization. However, these methods are not required for transformr models. Transformers are trained on raw text containing punctuations, emojis, non-alpha numerics, symbols, numbers, urls, hashtags and email formats. They use WordPiece or Byte-Pair Encoding to handle unknown words. They learn contextual embeddings, so stopwords carry meaningful context. These models learn from these patterns, especially when:
punctuation affects meaning ("Wow!" vs. "Wow."), emojis express sentiment, URLs or hashtags indicate context (e.g., in tweets or forum posts).

Unlike older methods (e.g., TF-IDF or RNNs), you do not need to remove stopwords, lemmatize/stem, manually lowercase (unless using an uncased model), clean punctuation (transformers learn from it!), one-hot encode or vectorize.

We will do the following text processing here.
1. Removing HTML tags and extra white spaces.
2. Removing duplicate entries
3. Remove Null values and replace them.

### Data Annotation
This involves classifying the data into genuine and fake based on various indicators derived from the raw text data

### Categorization of the data
The categories of the collected data is really ambiguous and needs to categorized well to understand the distribution.

For this project, we collected text data different News sources in Zambia using RSS feeds. Now we will clean, preprocess, annotate and categorize the data.

In [30]:
#import regex
import spacy
import re
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup as bf
from textblob import TextBlob
from urllib.parse import urlparse
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import KNeighborsClassifier

In [31]:
# Read the raw text data for clearning
df1 = pd.read_csv('../../data/raw-new.csv')
df2 = pd.read_csv('../../data/raw_old.csv')

In [32]:
df1.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [33]:
df2.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [34]:
df = pd.concat((df1, df2), axis=0)
df.shape

(16570, 7)

For text classification, the important columns are the headline and description. The source of the data and author information are important for classifying if the news is fake or not. We will also convert the date to standard pandas date format so that we can get an idea of when the news was published.

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16570 entries, 0 to 14343
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Source       16570 non-null  object
 1   Category     16096 non-null  object
 2   Headline     16569 non-null  object
 3   Link         16570 non-null  object
 4   Description  16550 non-null  object
 5   Date         16570 non-null  object
 6   Author       16569 non-null  object
dtypes: object(7)
memory usage: 1.0+ MB


In [36]:
df.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [37]:
# converting the date columnn to pandas date and time format
df['Date'] = pd.to_datetime(df['Date'])
df['Date'].iloc[:5]

0   2024-11-20 16:31:18+00:00
1   2024-11-20 15:15:28+00:00
2   2024-11-20 15:14:39+00:00
3   2024-11-20 15:13:41+00:00
4   2024-11-20 15:13:05+00:00
Name: Date, dtype: datetime64[ns, UTC]

In [38]:
# let's check for Nan values and duplicate entries
df.isna().sum()

Source           0
Category       474
Headline         1
Link             0
Description     20
Date             0
Author           1
dtype: int64

In [39]:
df[df['Category'].isna()]

,Source,Category,Headline,Link,Description,Date,Author
2013,Flava FM,NaN,Contact Us,https://flavaradioandtv.com/contact-us?utm_sou...,<p>Get In Touch Location: 3rd Floor Kitwe Main...,2021-01-19 08:43:25+00:00,NaN
2179,Copperbelt Energy,NaN,Businesses,https://cecinvestor.com/businesses/,Business segments Local Power Supply Source a...,2017-01-17 06:54:37+00:00,aiciadmin
2180,Copperbelt Energy,NaN,Management,https://cecinvestor.com/how-we-are-governed/ma...,How We Are Governed Management Most of CEC’s...,2017-01-15 06:01:43+00:00,aiciadmin
2181,Copperbelt Energy,NaN,Contact,https://cecinvestor.com/contact/,Investor Relations Precious M. Chisenga Corpor...,2017-01-12 06:10:53+00:00,aiciadmin
8546,Kitwe Online,NaN,TONGA LANGUAGE,https://kitweonline.com/languages/tonga-langua...,Resources: Here are some of the resources we h...,2022-10-20 10:57:00+00:00,JS
...,...,...,...,...,...,...,...
10670,Daily Nations Zambia,NaN,NaN,https://dailynationzambia.com/2021/03/10304/?u...,"Mon, 30 Nov -0001 00:00:00 +0000 WARRIORS HOLD...",2021-03-07 10:31:11+00:00,Daily Nation
10671,Daily Nations Zambia,NaN,FANS WANT POWER DYNAMOS HEAD COACH DISMISSED,https://dailynationzambia.com/2021/03/fans-wan...,"Mon, 30 Nov -0001 00:00:00 +0000 FANS WANT POW...",2021-03-07 10:31:11+00:00,Daily Nation
10672,Daily Nations Zambia,NaN,YOU CAN BUY THIS BOOK- A PRESIDENT BETRAYED FR...,https://dailynationzambia.com/2021/03/you-can-...,"Mon, 30 Nov -0001 00:00:00 +0000 Author: Richa...",2021-03-07 10:30:56+00:00,Daily Nation
10673,Daily Nations Zambia,NaN,"Phoenix gives KCC K32, 000 to help keep Kitwe ...",https://dailynationzambia.com/2021/03/phoenix-...,"Sat, 04 Feb 2017 11:29:22 +0000 By ROGERS KALE...",2021-03-07 10:30:56+00:00,Daily Nation


In [40]:
df[df['Description'].isna()]

,Source,Category,Headline,Link,Description,Date,Author
981,Kitwe Online,"Theatre,auditions,Kitwe,little theatre,mukonto...",Kitwe Little Theatre Auditions – June 2022,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2022-06-16 07:26:13+00:00,JS
993,Kitwe Online,"Art,Kitwe On Line,Self-Development,sotambe fil...",SOTAMBE FILM MAKING BOOTCAMP,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2021-02-18 13:05:02+00:00,JS
994,Kitwe Online,"History,Kitwe On Line,Kitwe,town,video",Kitwe Town Video,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2021-02-02 12:53:37+00:00,JS
995,Kitwe Online,"Kitwe On Line,liebherr",Wheeled Excavator For Sale,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2020-09-29 10:33:33+00:00,JS
1108,Kitwe Online,"books,Kitwe,kitweonline book store,zambia",Books,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2013-07-21 11:03:04+00:00,JS
1160,Kitwe Online,Classified Ads,Seasonal Greetings,https://kitweonline.com/search/kitwe/feed/rss2/,NaN,2012-12-22 23:24:01+00:00,JS
4100,Lusaka Times,Photo Gallery,2018 COSAFA U20 Championship in Pictures,https://www.lusakatimes.com/2018/12/15/2018-co...,NaN,2018-12-15 05:07:43+00:00,Chief Editor
4371,Lusaka Times,Videos and Audios,President Lungu’s Arrival in Kitwe,https://www.lusakatimes.com/2018/06/07/preside...,NaN,2018-06-07 04:45:30+00:00,Chief Editor
4849,Lusaka Times,Photo Gallery,The Week in Pictures,https://www.lusakatimes.com/2017/05/05/the-wee...,NaN,2017-05-05 11:04:35+00:00,editor
4967,Lusaka Times,Photo Gallery,President and Vice President Visits to Army Wo...,https://www.lusakatimes.com/2017/01/06/preside...,NaN,2017-01-06 14:41:41+00:00,editor


Okay. So there are plenty of null values in the category column and also the description ones.

In [41]:
# Look for duplicated entries
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
14339    False
14340    False
14341    False
14342    False
14343    False
Length: 16570, dtype: bool

In [42]:
# Let's drop the duplicates
df.drop_duplicates(inplace=True)
df.shape

(14282, 7)

In [43]:
# let's drop the rows which has nan values for the description
df.dropna(subset=["Description"], inplace=True)
df.shape

(14262, 7)

The 20 entries which did not have description and they are removed now.

In [44]:
df.isna().sum()

Source           0
Category       474
Headline         1
Link             0
Description      0
Date             0
Author           1
dtype: int64

In [45]:
# Fill out NaN values
df.fillna(value = "", inplace=True)

In [46]:
df.isna().sum()

Source         0
Category       0
Headline       0
Link           0
Description    0
Date           0
Author         0
dtype: int64

In [47]:
# Let's take a look at the unique sources, caegories of the news
df['Source'].value_counts()

Source
Lusaka Times               6472
Lusaka Voice               1744
Daily Nations Zambia       1361
Zambia Eye                 1022
Kitwe Online                877
Mwebantu                    711
Zambia Monitor              627
Copperbelt Energy           445
Zambian Eye                 300
Daily Revelation Zambia     202
Zambia365                   130
Zambia Reports              119
News Invasion 24             84
Lusaka Star                  67
ZNBC                         64
Flava FM                     14
DailyMail                    10
Tech Africa News              7
Christian Voice               4
Daily Mail Zambia             2
Name: count, dtype: int64

In [48]:
# Look at the unique categories
df['Category'].unique()[:50]

array(['Careers,Current Careers',
       'Corporate Announcements,Downloads,Featured,Green Bond', 'Careers',
       'Corporate Social Responsibility,Featured',
       'Corporate Announcements,Featured', 'Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured,AF',
       'Corporate Announcements,Featured,AF',
       'Corporate Announcements,Downloads,Featured,AF', 'News',
       'Corporate Social Responsibility,Featured,News,AF',
       'Corporate Social Responsibility,News',
       'Corporate Social Responsibility,Featured,News,Power Dynamos',
       'Corporate Social Responsibility,Power Dynamos',
       'Downloads,Featured,News', 'Careers,Featured,Power Dynamos',
       'Corporate Social Responsibility,News,Power Dynamos',
       'Corporate Announcements,Downloads,Featured', 'Power Dynamos',
       'Corporate Social Responsibility,Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured',
       'Featured,News,Project', 'Corporate Ann

It looks like the Category column consists of many entries, ambiguous and often mixed with headline column for many of the training data. Therefore it is important to come up with a unique set of categories and appending that information to the Category column. We will do the categorization of the data towards the end.

### Text Preprocessing with Spacy

In [49]:
# Loading the spacy english language model
# This model was trained on large corpus of labeled text data. Labels like sentence parsing, POS tagging, named entity recongnition 
nlp = spacy.load('en_core_web_sm')

### Cleaning of the data

In [54]:
# Function to remove html tags
def strip_html_tags(text):
    """
    Strip html tags in a text
    """
    soup = bf(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text

In [63]:
s = df['Description'].iloc[0]
print(s)

We currently have career opportunities in the following field VAC-2024-0026: GDP LEGAL Grade: CEC GDP | Contract Type: Fixed Term | Location: Kitwe Are you a fresh graduate with a Bachelor of Law Degree (LLB) and an Advocate of the High Court of Zambia; seeking to acquire real-world, hands-on experience to hone your strengths and


In [69]:
print(strip_html_tags(s))

We currently have career opportunities in the following field VAC-2024-0026: GDP LEGAL Grade: CEC GDP | Contract Type: Fixed Term | Location: Kitwe Are you a fresh graduate with a Bachelor of Law Degree (LLB) and an Advocate of the High Court of Zambia; seeking to acquire real-world, hands-on experience to hone your strengths and


In [71]:
# Let's make a copy of the dataset before doing the preprocessing
df_clean = df.copy()

# Now let's apply this function on the headline and the description column
df_clean['Headline'] = df_clean['Headline'].astype('str').apply(strip_html_tags)
df_clean['Description'] = df_clean['Description'].astype('str').apply(strip_html_tags)

/tmp/ipykernel_920369/1132794250.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bf(text, "html.parser")
/tmp/ipykernel_920369/1132794250.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bf(text, "html.parser")


In [72]:
df_clean.head(5)

,Source,Category,Headline,Link,Description,Date,Author
0,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: GDP – Legal,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 16:31:18+00:00,Lovejoy Musundire
1,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Assurance Specialist –...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:15:28+00:00,Lovejoy Musundire
2,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Internal Auditor – In...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:14:39+00:00,Lovejoy Musundire
3,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Engineer – Protection,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:41+00:00,Lovejoy Musundire
4,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Mechanic II,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:05+00:00,Lovejoy Musundire


### Label annotation
Now since we have finished the data cleaning process, the next important thing is to annotate the data as real or fake. We need to come up with a clear strategy of how to do that.
1. Checking if the source is a reliable and reputable news channel
2. Check if the news domain is suspicious
3. Look for clickbaits which has exaggerated use of sensational keywords
4. Check if the headline and description matches each other
5. check the polarity of the headline or the description, if its too negative
6. Check for execessive capitalization
7. Check for vague authors
8. Check for suspicious links

Let's classify the news as fake if it satisfies atleast 2 of these criterias.

In [76]:
class FakeNewsDetector():
    """
    Class to detect the genuinity of news elements
    within a given dataframe
    """

    def __init__(self, df):
        self.df = df # pandas data frame
        self.zambian_reputable_sources = ['daily-mail.co.zm', 'times.co.zm', 'znbc.co.zm', 'flavaradioandtv.com', 
            'lusakatimes.com', 'kitwetimes.com','zambiamonitor.com']
        self.suspicious_domain_pattern = re.compile(r'\\.(info|lo|ru|cn|xyz|top|news|live|buzz|click|online)$')
        
        # list of sensational words
        self.sensational_keywords = [
            'shocking', 'unbelievable', 'amazing', 'incredible', 'secret', 
            'exposed', 'you won’t believe', 'scandal', 'controversy'
        ]
    def check_vague_author(self, author):
        """
        Checking for vague authors
        """
        vague_authors = ['admin', 'editor', 'newsroom', 'staff', 'unknown']
        return 1 if any([vague_author in author.lower() for vague_author in vague_authors]) else 0
            
            
    def similarity_head_desc(self, row):
        """
        Try to find a cosine similarity between headline and
        description using tf-idf
        """
        comb_txt = [row['Headline'], row['Description']]
        # tfidf needs text without tokenizing as it will do the tokenizing, create the unique vocabulary and the vectorization
        
        tfidf_vectorizer = TfidfVectorizer()
        # This step will basically create a unique vocabulary and a matrix with features for each corpuse
        # elements. Or basically create a tf-idf vector for each row based on how frequency a token appears in one document and how the same 
        # appear in other documents
        tfidf_mat = tfidf_vectorizer.fit_transform(comb_txt)
        sim_score = cosine_similarity(tfidf_mat[0,:], tfidf_mat[1,:])

        return 1 if sim_score[0][0] < 0.10 else 0

    def check_source_credibility(self, url):
        """
        Check to see if the the netlocation of the URL
        is legit. For this parse the url and get the netloc infor. Then compare
        with predefined list
        """
        parsed_url = urlparse(url)
        domain = parsed_url.netloc.lower()
        return 1 if domain not in self.zambian_reputable_sources else 0
            
        
        
    def detect_clickbait(self, headline):
        """
        Look for excessive punctuations, all capital headlines
        Look for excessive usage of provocative words
        """
        excessive_punctuation = len(re.findall(r'[!?.]{2,}', headline)) > 0
        all_caps = headline.isupper()
        provocative_words = any(word in headline.lower() for word in [
            'shocking', 'unbelievable', 'you won’t believe', 'secret', 
            'amazing', 'incredible'
        ])
        return 1 if excessive_punctuation or all_caps or provocative_words else 0
    

    def count_sensational_keywords(self, description):
        """
        Count the number of sensational words in the description to make sure
        it's not filled with them
        """
        return sum(description.lower().count(word) for word in self.sensational_keywords)

    
    def get_sentiment_score(self, text):
        """
        Get the sentiment of the text if its positive or
        negative. Score goes from -1 to +1
        """
        try:
            sentiment = TextBlob(text).sentiment
            return 1 if abs(sentiment.polarity) > 0.5 else 0
        except Exception as e:
            return 0
            
    def check_excessive_capitalization(self, text):
        """
        Check for excessive capitalization in the 
        headline or description. Could be indication of fake news
        """
        text = text.split()
        capitalized_words = [word for word in text if word.isupper() and len(word) > 1]
        return 1 if len(capitalized_words) > 3 else 0 

    def count_suspicious_links(self, description):
        """
        Count the number of suspicious links in the description
        """
        urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\\\(\\\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', description)
        return 1 if len(urls) > 0 else 0
        
    def count_sensational_keywords(self, description):
        """
        Count how many keywords are in the description
        """
        
        return sum(description.lower().count(word) for word in self.sensational_keywords)
        
    def check_short_sensational_description(self, description):
        """
        Check for short sensational description
        """
        description_length = len(description)
        sensational_word_count = self.count_sensational_keywords(description)
        return 1 if description_length < 100 and sensational_word_count > 1 else 0

    def collect_fake_columns(self):
        
        # Fake author column
        self.df['fake_author'] = self.df['Author'].astype('str').apply(self.check_vague_author)

        # Similarity between source and descriptions
        self.df['dissimilar_head_desc'] = self.df.apply(self.similarity_head_desc, axis=1)

        # Legit URL netlocation
        self.df['fake_url'] = self.df['Link'].apply(self.check_source_credibility)

        # check for clickbaits
        self.df['click_bait'] = self.df['Headline'].astype('str').apply(self.detect_clickbait)
        #print(any(self.df['Headline'].apply(type) != 'str'))

        # sentiment score
        self.df['polar'] = self.df['Description'].astype('str').apply(self.get_sentiment_score)

        # excessive capitalization in the headline
        self.df['excess_capitalization'] = self.df['Headline'].astype('str').apply(self.check_excessive_capitalization)

        #Check for suspicious links
        self.df['susp_links'] = self.df['Description'].astype('str').apply(self.count_suspicious_links)

        # Check for short sensational description
        self.df['sens_description'] = self.df['Description'].astype('str').apply(self.check_short_sensational_description)
        
    def get_final_label(self):

        # replacing boolean with binary values
        #self.df.replace({True:1, False:0}, inplace=True)
        self.collect_fake_columns() # run the collection first
        self.df['fake_indicators'] = self.df.iloc[:,7:14].apply(np.sum, axis=1).astype('int')
        self.df['Target'] = self.df['fake_indicators'].copy()
        self.df['Target'] = self.df['Target'].apply(lambda x: 1 if x >=2 else 0)

In [77]:
ob = FakeNewsDetector(df_clean)
ob.get_final_label()

In [78]:
df_clean.iloc[200:222,:]

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
200,Copperbelt Energy,Corporate Announcements,CEC | Notice and Agenda of the Seventeenth Ann...,https://cecinvestor.com/search/kitwe/feed/rss2/,Notice is hereby given that the Seventeenth An...,2015-03-06 14:55:56+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
201,Copperbelt Energy,Tenders,CEC | Invitation to Tender in the CEC cost of...,https://cecinvestor.com/search/kitwe/feed/rss2/,The Copperbelt Energy Corporation PLC (CEC) is...,2015-02-02 09:09:58+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
202,Copperbelt Energy,Corporate Social Responsibility,CEC Renewable Energy Essay Writing Competition...,https://cecinvestor.com/search/kitwe/feed/rss2/,Copperbelt Energy Corporation Plc (CEC) is ple...,2015-01-15 13:55:19+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
203,Copperbelt Energy,Corporate Social Responsibility,CEC Renewable Energy Essay Writing Competition,https://cecinvestor.com/search/kitwe/feed/rss2/,Copperbelt Energy Corporation Plc (CEC) is an ...,2014-10-23 09:06:21+00:00,aiciadmin,1,1,1,0,0,0,0,0,3,1
204,Copperbelt Energy,Careers,Vacancy: IT Network Engineer,https://cecinvestor.com/search/kitwe/feed/rss2/,The Copperbelt Energy Corporation PLC (CEC) is...,2014-10-10 14:32:52+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
205,Copperbelt Energy,Careers,Vacancy: Management Accountant,https://cecinvestor.com/search/kitwe/feed/rss2/,The Copperbelt Energy Corporation PLC (CEC) is...,2014-09-25 15:29:05+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
206,Copperbelt Energy,Corporate Announcements,Sixteenth AGM presentation to CEC Shareholders,https://cecinvestor.com/search/kitwe/feed/rss2/,The Directors of Copperbelt Energy Corporation...,2014-07-30 15:29:01+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
207,Copperbelt Energy,Corporate Announcements,Notice of Sixteenth Annual General Meeting,https://cecinvestor.com/search/kitwe/feed/rss2/,Notice is hereby given that the Sixteenth Annu...,2014-07-11 16:14:31+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
208,Copperbelt Energy,News,CEC management commended by analysts,https://cecinvestor.com/search/kitwe/feed/rss2/,CEC has been commended by analysts for its pro...,2014-02-26 12:44:54+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
209,Copperbelt Energy,News,Reminder: CEC Investor Open Day,https://cecinvestor.com/search/kitwe/feed/rss2/,Copperbelt Energy Corporation PLC (CEC) invite...,2014-02-06 07:37:34+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1


In [79]:
# Checking number of news with fake authors
df_clean[df_clean['fake_author']==True].shape

(7294, 17)

In [80]:
# find instances where headline and description is not matching
df_clean[df_clean['dissimilar_head_desc']==True][:20]

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
18,Copperbelt Energy,Careers,CEC Career Opportunity: Instrumentation Techni...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC9 | Contract Type: Permanent | Locat...,2024-05-27 15:05:31+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
19,Copperbelt Energy,Careers,CEC Career Opportunity: Supply Assistant (01),https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC9 | Contract Type: Permanent | Locat...,2024-05-27 15:04:41+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
23,Copperbelt Energy,"Annual Report,Corporate Announcements,Download...",CEC 2023 Annual Report released,https://cecinvestor.com/search/kitwe/feed/rss2/,I am pleased with the great progress we made d...,2024-03-07 21:43:10+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
26,Copperbelt Energy,Careers,CEC Career Opportunity: Technician – Electrical,https://cecinvestor.com/search/kitwe/feed/rss2/,We invite applications from suitably qualified...,2024-01-24 15:37:07+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
27,Copperbelt Energy,Careers,CEC Career Opportunity: Manager – Digital Inno...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC5 | Contract Type: Permanent | Locat...,2023-11-30 06:04:12+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
28,Copperbelt Energy,Careers,CEC Career Opportunity: Engineer – Information...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC6 | Contract Type: Permanent | Locat...,2023-11-30 06:03:07+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
29,Copperbelt Energy,Careers,CEC Career Opportunity: Advisor – Talent Manag...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC6 | Contract Type: Permanent | Locat...,2023-10-02 19:00:24+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
30,Copperbelt Energy,Careers,CEC Career Opportunity: Advisor – HR Operation...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC6 | Contract Type: Permanent | Locat...,2023-10-02 18:52:36+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
31,Copperbelt Energy,Careers,CEC Career Opportunity: Manager – HR Operation...,https://cecinvestor.com/search/kitwe/feed/rss2/,Grade: CEC5 | Contract Type: Permanent | Locat...,2023-10-02 18:43:57+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
32,Copperbelt Energy,Careers,CEC Career Opportunity: Creditors Accountant,https://cecinvestor.com/search/kitwe/feed/rss2/,We invite applications from suitably qualified...,2023-09-06 06:56:20+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1


In [81]:
df_clean.head()

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
0,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: GDP – Legal,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
1,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Assurance Specialist –...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
2,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Internal Auditor – In...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
3,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Engineer – Protection,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
4,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Mechanic II,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0


### Categorization of the data
Most of the colletected data does not have a proper category and even for the ones with the category, they have been assigned wrong. Sometimes the category is filled with information from the description or ambiguous values.

Let's build a strategy for the categorization of each text sample.

In [82]:
class TextCategorizer:
    """
    A class to assign a set of pre-selected categories and 
    assign KNN based classifier to assign the nearest category using
    tf-idf vectorization
    """
    def __init__(self, data, n_neighbors=5, max_features=5000):
        self.data = data # pandas data frame
        self.n_neighbors = n_neighbors # no of nearest neighbours to consider
        self.max_features = max_features # maximum number of features to consider
        self.vectorizer = TfidfVectorizer(max_features=max_features) #tf-dif vectorizer
        self.knn = KNeighborsClassifier(n_neighbors=n_neighbors) # KNN classifier
        
        # Define category keywords directly in the class
        self.categories_keywords = {
            'sports': ['football', 'soccer', 'basketball', 'tennis', 'cricket', 'olympics', 'athlete', 'sports'],
            'politics': ['government', 'election', 'politician', 'policy', 'parliament', 'minister', 'president', 'vote'],
            'education': ['school', 'university', 'education', 'college', 'students', 'learning', 'teacher', 'scholarship'],
            'health and wellness': ['health', 'hospital', 'doctor', 'wellness', 'mental health', 'fitness', 'medicine', 'disease'],
            'development': ['development', 'infrastructure', 'construction', 'road', 'bridge', 'building', 'urbanization'],
            'narcotics': ['narcotics', 'drug', 'cocaine', 'heroin', 'meth', 'drug trafficking', 'illegal drugs'],
            'fashion': ['fashion', 'clothing', 'designer', 'runway', 'model', 'style', 'apparel', 'trends'],
            'career': ['job', 'career', 'employment', 'opportunity', 'work', 'recruitment', 'hiring', 'position'],
            'local news': ['local', 'community', 'city', 'town', 'village', 'municipality', 'neighborhood', 'region'],
            'economy news': ['economy', 'economic', 'finance', 'market', 'stocks', 'currency', 'inflation', 'gdp'],
            'business news': ['business', 'company', 'corporation', 'entrepreneur', 'startup', 'industry', 'investment', 'profit']
        }
        
    def prioritize_category(self, description):
        """Assign a single category based on highest keyword count."""

        keyword_count = {}
        for category, keywords in self.categories_keywords.items():
            count = sum(description.lower().count(keyword) for keyword in keywords)
            if count > 0:
                keyword_count[category] = count
        return max(keyword_count, key=keyword_count.get) if keyword_count else 'uncategorized'
    
    def assign_single_categories(self):
        """Apply single category based on keyword prioritization."""
        self.data['Single_Category'] = self.data['Description'].apply(self.prioritize_category)

    def train_knn_classifier(self):
        """Train the KNN model to predict categories for uncategorized entries."""
        desc = self.data['Description']
        
        cat = self.data['Single_Category'] != 'uncategorized'
        uncat = self.data['Single_Category'] == 'uncategorized'
        
        # Train data
        X_train = desc[cat]
        y_train = self.data['Single_Category'][cat]

        # test data
        X_test = desc[uncat]
        
        # Convert text to TF-IDF vectors
        X_train_tfidf = self.vectorizer.fit_transform(X_train)
        
        # Train KNN classifier
        self.knn.fit(X_train_tfidf, y_train)
        
        # Predict uncategorized entries
        if any(uncat):
            X_test_tfidf = self.vectorizer.transform(X_test) # vectorize the text to TF-IDF
            y_pred = self.knn.predict(X_test_tfidf)
            self.data.loc[uncat, 'Single_Category'] = y_pred
    
    def categorize(self):
        """Run all categorization steps in sequence, and replace 'Category' with 'Single_Category'."""
        self.assign_single_categories()
        self.train_knn_classifier()
        
        return self.data

In [83]:
tob = TextCategorizer(df_clean)
df_cat = tob.categorize()

In [84]:
df_cat.head()

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target,Single_Category
0,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: GDP – Legal,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,economy news
1,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Assurance Specialist –...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
2,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Internal Auditor – In...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
3,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Engineer – Protection,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
4,Copperbelt Energy,"Careers,Current Careers",CEC Career Opportunity: Mechanic II,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career


In [85]:
# Now drop the unnecessary columns and keep only the relevant ones
df_fin = df_cat.drop(columns=["Category", "fake_author", "dissimilar_head_desc", "fake_url", "click_bait", "polar", "excess_capitalization", "susp_links",
                     "sens_description", "fake_indicators"])

In [86]:
df_fin.head()

,Source,Headline,Link,Description,Date,Author,Target,Single_Category
0,Copperbelt Energy,CEC Career Opportunity: GDP – Legal,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,economy news
1,Copperbelt Energy,CEC Career Opportunity: Assurance Specialist –...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,career
2,Copperbelt Energy,CEC Career Opportunity: Internal Auditor – In...,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,career
3,Copperbelt Energy,CEC Career Opportunity: Engineer – Protection,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,career
4,Copperbelt Energy,CEC Career Opportunity: Mechanic II,https://cecinvestor.com/search/kitwe/feed/rss2/,We currently have career opportunities in the ...,2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,career


In [87]:
df_fin.rename(columns={"Single_Category":"Category"}, inplace=True)

In [88]:
# reordering the column names:
df_fin = df_fin[["Source", "Category", "Headline", "Description", "Link", "Date", "Author", "Target"]]

# print the head
df_fin.head()

,Source,Category,Headline,Description,Link,Date,Author,Target
0,Copperbelt Energy,economy news,CEC Career Opportunity: GDP – Legal,We currently have career opportunities in the ...,https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 16:31:18+00:00,Lovejoy Musundire,0
1,Copperbelt Energy,career,CEC Career Opportunity: Assurance Specialist –...,We currently have career opportunities in the ...,https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:15:28+00:00,Lovejoy Musundire,0
2,Copperbelt Energy,career,CEC Career Opportunity: Internal Auditor – In...,We currently have career opportunities in the ...,https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:14:39+00:00,Lovejoy Musundire,0
3,Copperbelt Energy,career,CEC Career Opportunity: Engineer – Protection,We currently have career opportunities in the ...,https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:13:41+00:00,Lovejoy Musundire,0
4,Copperbelt Energy,career,CEC Career Opportunity: Mechanic II,We currently have career opportunities in the ...,https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:13:05+00:00,Lovejoy Musundire,0


In [90]:
# Finally save the cleaned, annotated and categorized data
df_fin.to_csv("../../data/data-final-cleaned-llm.csv", index=False)

In [89]:
test = df_fin['Headline'].iloc[0]
print(test)

CEC Career Opportunity: GDP – Legal


In [47]:
# Also saving the data in pickle format to a preserve the data type structure
# df_fin.to_pickle("../../data/data-final-cleaned-llm.pkl")